In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, TimestampType


#Define expected schema

schema = StructType([
    StructField("user_id", StringType(), True),
    StructField("event_time", TimestampType(), True),
    StructField("event_type", StringType(), True),
    StructField("page", StringType(), True),
    StructField("device", StringType(), True)
])


df = (
    spark.readStream
    .format("cloudfiles")
    .option("cloudfiles.format", "json")
    .option("cloudfiles.schemaLocation", "/Volumes/retailnew/bronze/schema_logs/clickstream")
    .schema(schema)
    .load("/Volumes/retailnew/bronze/raw_data/clickstream")
    .withColumn("ingest_ts", F.current_timestamp())
       
)
(
    df.writeStream
    .format("Delta")
    .option("checkpointLocation", "/Volumes/retailnew/bronze/checkpoints/clickstream")
    .option("mergeSchema", "true")
    .trigger(availableNow=True)
    .table("retailnew.bronze.clickstream")
)